## Canopy Height Model (CHM) from UAV Lidar
### CHM = DSM (top surface) - DTM (bare earth ground).
### Represents height of vegetation / above-ground objects.

#### CRS: EPSG:25832.

In [ ]:

# USER CONFIGURATION

DSM_PATH    = "data/dsm.tif"          # <-- Digital Surface Model (top surface)
DTM_PATH    = "data/dtm.tif"          # <-- Digital Terrain Model (bare earth)
OUTPUT_PATH = "output/chm.tif"

# Cleaning parameters:
MIN_HEIGHT     = 0.0      # clamp tiny negatives (DSM<DTM noise) to 0
MAX_HEIGHT     = 50.0     # cap unrealistic heights (blunders)
SMOOTH_PITS    = True     # fill single-pixel "data pits" with local median
PIT_WINDOW     = 3        # window (pixels) used for pit filling

In [ ]:
# Import required Libraries

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from scipy import ndimage
from dem_utils import read_dem, write_raster

In [ ]:
## Load and align DSM & DTM
# The DSM is resampled onto the DTM grid so pixels line up exactly.

with rasterio.open(DTM_PATH) as ref:
    dtm = ref.read(1).astype("float64")
    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_nodata = ref.nodata
    res_x, res_y = ref.res

    if ref_crs is None:
        print("WARNING: DTM has no CRS. Assuming EPSG:25832.")
    elif ref_crs.to_epsg() != 25832:
        print(f"WARNING: DTM CRS is {ref_crs}. Expected EPSG:25832.")

if ref_nodata is not None:
    dtm[dtm == ref_nodata] = np.nan

# Reproject / resample DSM to the DTM reference grid
dsm = np.empty_like(dtm)
with rasterio.open(DSM_PATH) as src:
    reproject(
        source=rasterio.band(src, 1),
        destination=dsm,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        resampling=Resampling.bilinear,
        dst_nodata=np.nan,
    )

print(f"Grid shape: {dtm.shape}, pixel size: {res_x} x {res_y} m")

In [ ]:
## Compute CHM

chm = dsm - dtm

# Clamp small negatives caused by interpolation/registration noise
if MIN_HEIGHT is not None:
    chm = np.where(chm < MIN_HEIGHT, MIN_HEIGHT, chm)

# Remove unrealistically high blunders
if MAX_HEIGHT is not None:
    chm = np.where(chm > MAX_HEIGHT, np.nan, chm)

In [ ]:
# Fill single-pixel data pits
# Replaces isolated invalid/low cells with the local median.

def fill_pits(arr, window=3):
    """Replace NaN cells with the local median from a moving window."""
    mask = np.isnan(arr)
    if not mask.any():
        return arr
    filled = arr.copy()
    median = ndimage.generic_filter(
        np.where(mask, np.nan, arr),
        function=np.nanmedian,
        size=window,
        mode="nearest",
    )
    filled[mask] = median[mask]
    return filled

if SMOOTH_PITS:
    chm = fill_pits(chm, PIT_WINDOW)

In [ ]:
# Summary statistics

valid = chm[~np.isnan(chm)]
print("=== CHM SUMMARY ===")
print(f"Valid cells   : {valid.size:,}")
print(f"Min height    : {np.nanmin(chm):.2f} m")
print(f"Max height    : {np.nanmax(chm):.2f} m")
print(f"Mean height   : {np.nanmean(chm):.2f} m")
print(f"95th pct      : {np.nanpercentile(chm, 95):.2f} m")

In [ ]:
# Visualize the CHM

vmax = np.nanpercentile(chm, 98)
plt.figure(figsize=(10, 8))
im = plt.imshow(chm, cmap="Greens", vmin=0, vmax=vmax)
plt.colorbar(im, label="Canopy height [m]")
plt.title("Canopy Height Model (CHM)")
plt.axis("off")
plt.show()

# histogram of heights
plt.figure(figsize=(8, 4))
plt.hist(valid, bins=50, color="forestgreen", edgecolor="k", alpha=0.7)
plt.xlabel("Canopy height [m]")
plt.ylabel("Cell count")
plt.title("CHM Height Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Save CHM raster
write_raster(OUTPUT_PATH, chm, ref_profile)